# QCoDeS example with Menlo Micro MM4250

The [Menlo Micro MM4250](https://menlomicro.com/products/rf-switches) is a single-pole six-throw (SP6T) RF MEMS switch, DC to 10 GHz, rated down to 10 mK. Besides the six RF ports it can connect the common port (RFC) to an internal LOAD or SHORT standard, which is useful for calibrating inside a cryostat. It is controlled through Menlo's **USB HiV Driver Board**, which appears as a USB HID device (no VISA address, no serial port).

This notebook covers finding and connecting to the board, selecting RF channels, the internal calibration states, the settle time, reconnecting without disturbing the switch, and recording the switch position with your data.

### Safety

- **This drives real hardware.** Every cell that sets `channel` or `state` physically moves the switch.
- **No hot switching.** The MM4250 is not rated for switching with RF power or a bias applied. Turn your source off (or below about 0.5 V) before changing position. See "Hot Switch Restrictions" in the datasheet and the pattern under *Using the switch in a measurement* below.
- **Both outputs move together.** The board has two Micro-D HV outputs (J19 and J18) and the driver writes the same position to both, so if two MM4250 modules hang off the split cable, both switch.

### Before running

- `pip install hidapi`. Not `hid`, which is a different package with a same-named module.
- The HiV Driver Board is plugged in over USB and powered.
- No other kernel has the switch open. The board takes an exclusive USB handle.

## Imports

In [ ]:
from qcodes.station import Station

from qcodes_contrib_drivers.drivers.MenloMicro import MenloMicroMM4250, SP6TState

## Find the driver board

`list_connected_boards()` lists every HiV Driver Board on this computer. You only need it when more than one is plugged in: pass the `serial_number` of the one you want to the constructor.

In [ ]:
MenloMicroMM4250.list_connected_boards()

## Connect

Opens the driver board over USB and prints the standard QCoDeS connect message.

Connecting **forces `ALL_OPEN`** by default, so the switch always starts from a known, de-energized state. To reconnect without moving the switch, see *Reconnecting mid-experiment* below.

In [ ]:
switch = MenloMicroMM4250("switch")

## What the driver exposes

Two parameters give the switch position, both views of the same thing:

- `state`: the full position: `"ALL_OPEN"`, `"RFC_RF1"`...`"RFC_RF6"`, `"INTERNAL_LOAD"` or `"INTERNAL_SHORT"`.
- `channel`: the friendlier 1-6 view, matching the RF1-RF6 SMA ports. It is `None` when the switch is not on a numbered port, as it is right now on `ALL_OPEN`.

Getting either one reads the position back from the board rather than returning what was last set.

A third, `settle_time`, is how long the driver waits after each switch command (see *Settle time* below).

The `IDN` serial is the driver board's USB serial number, not the serial of the switch module itself.

In [ ]:
switch.print_readable_snapshot(update=True)

## Select an RF channel

`switch.channel(n)` connects RFC to RFn. Reading `channel` or `state` afterwards confirms the board actually took the command.

In [ ]:
switch.channel(3)
print("channel:", switch.channel())
print("state:  ", switch.state())

## Step through all six channels

Set each channel and verify the readback. The driver waits `settle_time` after each command before returning.

In [ ]:
for ch in range(1, 7):
    switch.channel(ch)
    readback = switch.channel()
    assert readback == ch, f"asked for RF{ch}, board reports {readback}"
    print(f"RF{ch}: ok ({switch.state()})")

## Internal calibration states

`ALL_OPEN`, `INTERNAL_LOAD` and `INTERNAL_SHORT` aren't numbered ports, so they're only reachable through `state`. On one of them, `channel` returns `None`.

In [ ]:
switch.state("INTERNAL_LOAD")
print("state:  ", switch.state())
print("channel:", switch.channel())

You can also pass the `SP6TState` enum instead of a string, which lets your editor autocomplete the names:

In [ ]:
switch.state(SP6TState.INTERNAL_SHORT)
switch.state()

## Which HV lines are energized

`active_hv_lines()` lists the driver-board HV lines that should be energized for the current position, per the datasheet. Useful when checking the board's outputs with a meter or scope. It uses the last known `state` rather than querying the board.

In [ ]:
switch.channel(1)
switch.active_hv_lines()

## Settle time

After each switch command the driver waits `settle_time` seconds before returning. The default, 25 ms, is the datasheet switching time and what Menlo's cryogenic app note recommends waiting at temperature. Raise it if your switch actuates more slowly, for example with long or filtered gate lines. You can also set it at connect time with `MenloMicroMM4250("switch", settle_time=...)`.

In [ ]:
print("default:", switch.settle_time(), "s")
switch.settle_time(0.05)
switch.channel(2)
print("now:    ", switch.settle_time(), "s")
switch.settle_time(0.025)

## Bad inputs never reach the hardware

Out-of-range or non-integer channels and misspelled states are rejected by the parameter validators before anything is sent to the board.

In [ ]:
for bad in [lambda: switch.channel(7), lambda: switch.channel(2.5), lambda: switch.state("RFC_RF7")]:
    try:
        bad()
    except (ValueError, TypeError) as err:
        print("rejected:", err.args[0])

print("still on:", switch.state())

## Reconnecting mid-experiment

If you close the switch or restart the kernel partway through, reconnect with `reset_on_connect=False` to leave the switch where it is. The driver reads the current position from the board instead of forcing `ALL_OPEN`.

In [ ]:
switch.channel(5)
switch.close()

switch = MenloMicroMM4250("switch", reset_on_connect=False)
print("channel after reconnect:", switch.channel())

Creating a second instrument with a name that's already in use raises an error **without** touching the switch, so accidentally re-running the connect cell can't reset a sweep in progress:

In [ ]:
try:
    MenloMicroMM4250("switch")
except KeyError as err:
    print("refused:", err)

print("still on:", switch.state())

## Recording the switch position with your data

Add the switch to a QCoDeS `Station`. Every dataset measured with that station saves a snapshot of it, so each run records which position it was taken on without you logging it by hand.

In [ ]:
station = Station(switch)
station.snapshot(update=True)["instruments"]["switch"]["parameters"]["state"]["value"]

## Using the switch in a measurement

In a real sweep, turn the source off around every switch move so the switch never changes position under RF power. With a Keysight PNA-family VNA (QCoDeS's `KeysightPNABase` drivers, whose `output` parameter turns the RF source on and off) the pattern looks like this:

```python
for ch in range(1, 7):
    vna.output(False)          # no hot switching
    switch.channel(ch)         # returns after settle_time
    assert switch.channel() == ch
    vna.output(True)
    # ... measure, e.g. with Measurement or do0d, using a Station that includes `switch`
vna.output(False)
switch.open_all()
```

## Clean up

Return the switch to `ALL_OPEN` and release the USB handle so other kernels can connect.

In [ ]:
switch.open_all()
switch.close()